# Classification: Regularized Logistic Regression for Life Insurance Underwriting

This notebook applies logistic regression with L1, L2, and Elastic Net regularization to the Prudential Life Insurance Assessment dataset. Instead of predicting the full 8-level risk rating, we frame underwriting as a binary decision: whether an application is assigned rating class 8 (the most common class) or not.

Regularization is central to this problem. The dataset has over 120 correlated features, including dozens of sparse medical keyword flags, so we compare how each penalty shrinks coefficients, which features it selects, and how stable those selections are.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibrationDisplay
from sklearn.metrics import roc_auc_score, brier_score_loss, confusion_matrix, classification_report, ConfusionMatrixDisplay, RocCurveDisplay

The dataset comes from Kaggle's Prudential Life Insurance Assessment competition. Only `train.csv` is used: the competition's `test.csv` has no `Response` labels, so all evaluation uses a stratified 80/20 split of the labeled training data.

Link: https://www.kaggle.com/c/prudential-life-insurance-assessment

**Running in Colab:** upload `train.csv` through the Files panel, or run the next cell and choose the file when prompted.

In [ ]:
# loading dataset (works locally and in Colab)
DATA_PATH = "data/train.csv"

if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        if not os.path.exists("train.csv"):
            files.upload()
        DATA_PATH = "train.csv"
    except ImportError:
        raise FileNotFoundError("Place train.csv inside the data/ folder.")

df = pd.read_csv(DATA_PATH)
df.shape

### Preliminaries

In [ ]:
df.head()

In [ ]:
df.info()

Several features have missing values. We check the share of missing entries per column, since this affects both imputation and which columns may be dropped under the 20% feature rule.

In [ ]:
# percentage of missing values per column
missing_pct = df.isna().mean().mul(100).sort_values(ascending=False)
missing_pct[missing_pct > 0].round(2)

The original target `Response` is an ordinal risk rating from 1 to 8. We first look at its distribution.

In [ ]:
# original label - ordinal rating from 1 to 8
df["Response"].value_counts().sort_index()

In [ ]:
df["Response"].value_counts().sort_index().plot(kind="bar", color="steelblue")
plt.xlabel("Response (rating class)")
plt.ylabel("Count")
plt.title("Distribution of the original Response variable")
plt.show()

We convert the target into a binary label: `1` if the application was assigned rating class 8, and `0` for classes 1 to 7. Prudential does not document whether class 8 represents the lowest or highest risk, so we describe it neutrally as the most common rating class.

In [ ]:
# binary label - 1 for rating class 8, 0 for classes 1 to 7
y = df["Response"].map({1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0, 8: 1})
y.value_counts()

**Feature reduction log (20% rule).** The project allows dropping at most 20% of the original features. `Id` is an identifier rather than a feature, and `Response` is the target, so neither counts toward the limit. Every dropped column is listed below with its reason.

In [ ]:
feature_cols = [c for c in df.columns if c not in ["Id", "Response"]]
max_drop = int(0.20 * len(feature_cols))
print(f"Original features: {len(feature_cols)} | Maximum columns we may drop: {max_drop}")

# candidates: features with more than 30% missing values
print("Features with >30% missing:", list(missing_pct[missing_pct > 30].index))

# TODO: finalize after EDA - format {column: reason}
cols_to_drop = {}

assert len(cols_to_drop) <= max_drop, "Dropped too many columns under the 20% rule"
print(f"Dropping {len(cols_to_drop)} of {max_drop} allowed columns")

In [ ]:
# features
X = df[feature_cols].drop(columns=list(cols_to_drop))
X.shape

Splitting the dataset with `stratify=y` ensures both the train and test sets preserve the class distribution of the binary label.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### Preprocessing

Unlike the class example, this dataset has missing values and one text-based categorical column (`Product_Info_2`). We handle both inside the pipeline, so every transformation is learned from the training data only:
- **Numeric features:** fill missing values with the median, then standardize.
- **Categorical features:** one-hot encode.

In [ ]:
categorical_cols = X.select_dtypes(include="object").columns.tolist()
numeric_cols = X.select_dtypes(exclude="object").columns.tolist()

preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

### Evaluation Helper

`evaluate()` runs the same checks used in the class notebook for any fitted model: the classification report, the confusion matrix, the ROC-AUC, and the ROC curve. It also stores each model's ROC-AUC and Brier score in `results` so we can compare all models at the end.

In [ ]:
LABELS = ["Class 1-7", "Class 8"]
results = {}

def evaluate(model, name):
    """Print the classification report, confusion matrix, and ROC curve of a fitted model,
    and save its test ROC-AUC and Brier score for the final comparison."""
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print(f"=== {name} ===")
    print(classification_report(y_test, y_pred, target_names=LABELS))
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=LABELS, cmap="Blues")
    plt.title(name)
    plt.show()

    auc = roc_auc_score(y_test, y_prob)
    print("ROC-AUC:", auc)
    RocCurveDisplay.from_predictions(y_test, y_prob, name=name)
    plt.show()

    results[name] = {"ROC-AUC": auc, "Brier score": brier_score_loss(y_test, y_prob)}
    return y_pred, y_prob

### Baseline Logistic Regression

The baseline model has no penalty. It serves as the reference point for measuring what regularization adds.

In [ ]:
base_model = Pipeline([
    ("preprocess", preprocess),
    ("logreg", LogisticRegression(penalty=None, max_iter=1000))
])

base_model.fit(X_train, y_train)
y_pred, y_prob = evaluate(base_model, "Baseline (no penalty)")

### Logistic Regression with L2 regularization

L2 (ridge) regularization shrinks all coefficients toward zero without removing any. The regularization strength `C` (the inverse of λ) is tuned with 5-fold stratified cross-validation on the training set, using ROC-AUC as the scoring metric.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
C_grid = [0.001, 0.01, 0.1, 1, 10]

l2_pipeline = Pipeline([
    ("preprocess", preprocess),
    ("logreg", LogisticRegression(penalty="l2", max_iter=1000))
])

l2_search = GridSearchCV(l2_pipeline, {"logreg__C": C_grid}, cv=cv, scoring="roc_auc", n_jobs=-1)
l2_search.fit(X_train, y_train)
print("Best C:", l2_search.best_params_, "| CV ROC-AUC:", round(l2_search.best_score_, 4))

l2_y_pred, l2_y_prob = evaluate(l2_search.best_estimator_, "L2 (Ridge)")

### Logistic Regression with L1 regularization

L1 (lasso) regularization can shrink coefficients exactly to zero, so it also performs feature selection. We report how many features the tuned model keeps.

In [ ]:
l1_pipeline = Pipeline([
    ("preprocess", preprocess),
    ("logreg", LogisticRegression(penalty="l1", solver="saga", max_iter=5000))
])

l1_search = GridSearchCV(l1_pipeline, {"logreg__C": C_grid}, cv=cv, scoring="roc_auc", n_jobs=-1)
l1_search.fit(X_train, y_train)
print("Best C:", l1_search.best_params_, "| CV ROC-AUC:", round(l1_search.best_score_, 4))

l1_y_pred, l1_y_prob = evaluate(l1_search.best_estimator_, "L1 (Lasso)")

In [ ]:
# features kept by the tuned L1 model
l1_best = l1_search.best_estimator_
l1_coefs = l1_best.named_steps["logreg"].coef_.ravel()
feature_names = l1_best.named_steps["preprocess"].get_feature_names_out()
print(f"Non-zero coefficients: {(l1_coefs != 0).sum()} of {len(l1_coefs)}")

### Logistic Regression with Elastic Net regularization

Elastic Net mixes the L1 and L2 penalties. Besides `C`, we tune `l1_ratio`, which sets the share of the L1 penalty. This grid takes longer to run because it fits more models.

In [ ]:
en_pipeline = Pipeline([
    ("preprocess", preprocess),
    ("logreg", LogisticRegression(penalty="elasticnet", solver="saga", max_iter=5000))
])

en_grid = {"logreg__C": [0.01, 0.1, 1], "logreg__l1_ratio": [0.2, 0.5, 0.8]}
en_search = GridSearchCV(en_pipeline, en_grid, cv=cv, scoring="roc_auc", n_jobs=-1)
en_search.fit(X_train, y_train)
print("Best params:", en_search.best_params_, "| CV ROC-AUC:", round(en_search.best_score_, 4))

en_y_pred, en_y_prob = evaluate(en_search.best_estimator_, "Elastic Net")

### Regularization Paths

A regularization path shows how each coefficient changes as `C` varies. Features that stay non-zero under strong L1 penalties are the most informative.

In [ ]:
# TODO: refit the L1 pipeline over a fine grid of C values,
# store coef_ at each C, and plot coefficient values against log(C)

### Stability Selection

A single L1 fit can select different features on slightly different data. Here we refit the L1 model on many bootstrap samples and record how often each feature is selected.

In [ ]:
# TODO: for B bootstrap samples of the training set, fit the tuned L1 pipeline,
# record which coefficients are non-zero, and plot the top features by selection frequency

### Calibration and Threshold

An insurer needs probabilities it can trust, not just labels. We check how well the predicted probabilities match observed frequencies, then choose a decision threshold based on misclassification costs instead of the default 0.5.

In [ ]:
# TODO: CalibrationDisplay for the tuned models,
# then compare costs across thresholds using a stated cost ratio for false positives vs false negatives

### Model Comparison and Conclusion

In [ ]:
pd.DataFrame(results).T.round(4)

*TODO: summarize which penalty performed best, which features were consistently selected, and how the results relate to underwriting.*